In [1]:
raw_users = [
("U001","Amit","29","Hyderabad","50000"),
("U002","Neha","Thirty Two","Delhi","62000"),
("U003","Ravi",None,"Bangalore","45k"),
("U004","Pooja","28","Mumbai",58000),
("U005",None,"31","Chennai","")
]

In [2]:
from pyspark.sql.types import *

schema_users = StructType([
    StructField("user_id", StringType()),
    StructField("name", StringType()),
    StructField("age", StringType()),
    StructField("city", StringType()),
    StructField("salary", StringType())
])

In [12]:
from pyspark.sql.functions import *
from pyspark.sql import SparkSession

# Initialize SparkSession
spark = SparkSession.builder.appName("Colab PySpark").getOrCreate()

df = spark.createDataFrame(raw_users, schema_users)

df = df.withColumn("age_int", when(col("age").rlike("^[0-9]+$"), col("age").cast("int")).otherwise(None))

df = df.withColumn(
    "salary_int",
    when(col("salary").rlike("k"),
         regexp_replace(col("salary"), "k", "").cast("int") * 1000)
    .when(col("salary") == "", None)
    .otherwise(col("salary").cast("int"))
)

df = df.fillna({"name": "UNKNOWN"})

final_users = df.filter(col("age_int").isNotNull()) \
                .select("user_id","name","age_int","city","salary_int")

final_users.show()

+-------+-------+-------+---------+----------+
|user_id|   name|age_int|     city|salary_int|
+-------+-------+-------+---------+----------+
|   U001|   Amit|     29|Hyderabad|     50000|
|   U004|  Pooja|     28|   Mumbai|     58000|
|   U005|UNKNOWN|     31|  Chennai|      NULL|
+-------+-------+-------+---------+----------+



In [9]:
raw_orders = [
("O001","U001","Laptop,Mobile,Tablet",75000),
("O002","U002",["Mobile","Tablet"],32000),
("O003","U003","Laptop",72000),
("O004","U004",None,25000),
("O005","U005","Laptop|Mobile",68000)
]


In [8]:
schema_orders = StructType([
    StructField("order_id", StringType()),
    StructField("user_id", StringType()),
    StructField("items", StringType()),
    StructField("amount", IntegerType())
])

In [10]:
df = spark.createDataFrame(raw_orders, schema_orders)

df = df.withColumn(
    "items_array",
    when(col("items").isNull(), array())
    .otherwise(split(regexp_replace(col("items"), "\\|", ","), ","))
)

exploded = df.withColumn("item", explode("items_array"))

exploded.groupBy("item").count().show()

+--------+-----+
|    item|count|
+--------+-----+
| [Mobile|    1|
|  Laptop|    3|
|  Mobile|    2|
|  Tablet|    1|
| Tablet]|    1|
+--------+-----+



In [14]:
raw_devices = [
    ("U001",{"mobile":120,"laptop":300}),
    ("U002","mobile:200,tablet:100"),
    ("U003",{"desktop":"400","mobile":"150"}),
    ("U004",None),
    ("U005","laptop-250")
]

schema = StructType([
    StructField("user_id", StringType()),
    StructField("usage", StringType())
])

df = spark.createDataFrame(raw_devices, schema)

df = df.withColumn(
    "usage_map",
    when(col("usage").isNull(), create_map())
    .otherwise(
        map_from_entries(
            transform(
                split(regexp_replace(col("usage"), "-", ":"), ","),
                lambda x: struct(
                    split(x, ":").getItem(0),
                    when(size(split(x, ":")) > 1, split(x, ":").getItem(1).cast("int")).otherwise(None)
                )
            )
        )
    )
)

df = df.withColumn("mobile_usage", col("usage_map").getItem("mobile"))

df.filter(col("mobile_usage") > 150).show()

+-------+--------------------+--------------------+------------+
|user_id|               usage|           usage_map|mobile_usage|
+-------+--------------------+--------------------+------------+
|   U002|mobile:200,tablet...|{mobile -> 200, t...|         200|
+-------+--------------------+--------------------+------------+



In [15]:
raw_profiles = [
("U001","Hyderabad,Telangana,500081"),
("U002",{"city":"Delhi","state":"Delhi","pincode":"110001"}),
("U003",("Bangalore","Karnataka",560001)),
("U004","Mumbai,MH"),
("U005",None)
]

In [17]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import col, when, lit, struct, split


processed_raw_profiles = []
for user_id, address_data in raw_profiles:
    if isinstance(address_data, (dict, tuple)):
        processed_raw_profiles.append((user_id, None))
    else:
        processed_raw_profiles.append((user_id, address_data))


profile_schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("address", StringType(), True)
])
df = spark.createDataFrame(processed_raw_profiles, schema=profile_schema)


df = df.withColumn(
    "addr",
    when(col("address").isNull(), struct(lit(None).alias("city"), lit(None).alias("state"), lit(None).alias("pincode")))
    .when(col("address").rlike("^[a-zA-Z\\s]+,[a-zA-Z\\s]+,[0-9]+$"),
          struct(
              split(col("address"), ",")[0].alias("city"),
              split(col("address"), ",")[1].alias("state"),
              split(col("address"), ",")[2].cast("int").alias("pincode")
          ))
    .when(col("address").rlike("^[a-zA-Z\\s]+,[a-zA-Z\\s]+$"),
          struct(
              split(col("address"), ",")[0].alias("city"),
              split(col("address"), ",")[1].alias("state"),
              lit(None).cast(IntegerType()).alias("pincode")
          ))
    .otherwise(struct(lit(None).alias("city"), lit(None).alias("state"), lit(None).alias("pincode")))
)

df.select(
    "user_id",
    col("addr.city"),
    col("addr.state"),
    col("addr.pincode")
).show()

+-------+---------+---------+-------+
|user_id|     city|    state|pincode|
+-------+---------+---------+-------+
|   U001|Hyderabad|Telangana| 500081|
|   U002|     NULL|     NULL|   NULL|
|   U003|     NULL|     NULL|   NULL|
|   U004|   Mumbai|       MH|   NULL|
|   U005|     NULL|     NULL|   NULL|
+-------+---------+---------+-------+



In [18]:
raw_transactions = [
("T001","2024-01-05","45000"),
("T002","05/01/2024",52000),
("T003","Jan 06 2024","Thirty Thousand"),
("T004",None,38000),
("T005","2024/01/07","42000")
]

In [24]:
from pyspark.sql.functions import col, coalesce, to_date, cast, when, lit

df = spark.createDataFrame(raw_transactions, ["txn_id","date","amount"])

# Define regex patterns for each date format to validate before parsing
date_format_yyyy_MM_dd = r"^\\d{4}-\\d{2}-\\d{2}$"
date_format_dd_MM_yyyy = r"^\\d{2}/\\d{2}/\\d{4}$"
date_format_MMM_dd_yyyy = r"^(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec) \\d{2} \\d{4}$"
date_format_yyyy_MM_dd_slash = r"^\\d{4}/\\d{2}/\\d{2}$"

df = df.withColumn(
    "txn_date",
    when(col("date").rlike(date_format_yyyy_MM_dd), to_date(col("date"), "yyyy-MM-dd"))
    .when(col("date").rlike(date_format_dd_MM_yyyy), to_date(col("date"), "dd/MM/yyyy"))
    .when(col("date").rlike(date_format_MMM_dd_yyyy), to_date(col("date"), "MMM dd yyyy"))
    .when(col("date").rlike(date_format_yyyy_MM_dd_slash), to_date(col("date"), "yyyy/MM/dd"))
    .otherwise(lit(None).cast("date"))
)

df = df.withColumn("amount_int", when(col("amount").rlike("^[0-9]+$"), col("amount").cast("int")).otherwise(None))

valid = df.filter(col("txn_date").isNotNull() & col("amount_int").isNotNull())
invalid = df.subtract(valid)

valid.show()
invalid.show()

+------+----+------+--------+----------+
|txn_id|date|amount|txn_date|amount_int|
+------+----+------+--------+----------+
+------+----+------+--------+----------+

+------+-----------+---------------+--------+----------+
|txn_id|       date|         amount|txn_date|amount_int|
+------+-----------+---------------+--------+----------+
|  T002| 05/01/2024|          52000|    NULL|     52000|
|  T001| 2024-01-05|          45000|    NULL|     45000|
|  T004|       NULL|          38000|    NULL|     38000|
|  T005| 2024/01/07|          42000|    NULL|     42000|
|  T003|Jan 06 2024|Thirty Thousand|    NULL|      NULL|
+------+-----------+---------------+--------+----------+

